In [19]:
import subprocess
import os
import csv
import pandas as pd
from tqdm import tqdm
from datetime import datetime
from pathlib import Path

In [20]:
os.chdir("C:/Users/ed/Development/dhzw/8-sensitivity-analysis")

In [21]:
# function to print elapsed time
def calculate_elapsed_time(start_time, end_time):
    elapsed_time = end_time - start_time
    total_seconds = elapsed_time.total_seconds()
    seconds = int(total_seconds % 60)
    total_minutes = total_seconds // 60
    minutes = int(total_minutes % 60)
    hours = int(total_minutes // 60)
    
    seconds = str(seconds).zfill(2)
    minutes = str(minutes).zfill(2)
    hours = str(hours).zfill(2)

    return str(hours) + ':' + str(minutes) + ':' + str(seconds)

# Initialise

### Read parameter sets to simulate

In [22]:
# read parameters to run
table = pd.read_csv('data/parametersets_to_run.csv')
table

,alphaWalk,alphaBike,alphaCarDriver,alphaCarPassenger,alphaBus,alphaTrain,betaTimeWalk,betaTimeBike,betaTimeCarDriver,betaTimeCarPassenger,...,betaCostCarPassenger,betaCostBus,betaCostTrain,betaTimeWalkTransport,betaChangesTransport,used_parameter_label,used_parameter_value,carCostKm,ptCostKm,ptBaseCost
0,-10.0,0.0,0.0,0.0,0.0,0.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaWalk,-10.0,0.25,0.187,1.08
1,-8.0,0.0,0.0,0.0,0.0,0.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaWalk,-8.0,0.25,0.187,1.08
2,-6.0,0.0,0.0,0.0,0.0,0.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaWalk,-6.0,0.25,0.187,1.08
3,-4.0,0.0,0.0,0.0,0.0,0.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaWalk,-4.0,0.25,0.187,1.08
4,-2.0,0.0,0.0,0.0,0.0,0.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaWalk,-2.0,0.25,0.187,1.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,0.0,0.0,0.0,0.0,0.0,2.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaTrain,2.0,0.25,0.187,1.08
62,0.0,0.0,0.0,0.0,0.0,4.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaTrain,4.0,0.25,0.187,1.08
63,0.0,0.0,0.0,0.0,0.0,6.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaTrain,6.0,0.25,0.187,1.08
64,0.0,0.0,0.0,0.0,0.0,8.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaTrain,8.0,0.25,0.187,1.08


### Initialise empty the output dataframe

In [23]:
# Initialise dataframe for mode choice distributions (1 line = 1 distribution = 1 iteration)
df = pd.DataFrame(columns=["CAR_DRIVER", "CAR_PASSENGER", "BIKE", "BUS_TRAM", "TRAIN", "WALK"])

for i in range(0, len(table)):
    df.to_csv("output/iterations/output_proportions_" + str(i) + ".csv", index=False)

In [24]:

def intialise_java():

    jdk_11_path = r"C:\Program Files\Java\jdk-11"

    os.environ["JAVA_HOME"] = jdk_11_path
    bin_path = os.path.join(jdk_11_path, "bin")
    os.environ["PATH"] = bin_path + os.pathsep + os.environ.get("PATH", "")

    # 4. Verification
    try:
        result = subprocess.run(["java", "-version"], capture_output=True, text=True, check=True)
        print("Java Version Output:\n", result.stderr) 
    except subprocess.CalledProcessError as e:
        print(f"Error: {e}")
    except FileNotFoundError:
        print("Java executable not found in the updated PATH.")

intialise_java()


Java Version Output:
 openjdk version "11.0.20.1" 2023-08-24 LTS
OpenJDK Runtime Environment Microsoft-8297088 (build 11.0.20.1+1-LTS)
OpenJDK 64-Bit Server VM Microsoft-8297088 (build 11.0.20.1+1-LTS, mixed mode)



# Call Sim2APL iteratively

### Prepare the Java command

In [25]:
# baseline of the Java command
config = "--config src/main/resources/config_DHZW_stt_senstivity.toml"


output = "-o output/deskrun"
parametersPath = "--parameter_file ../8-sensitivity-analysis/data/parametersets_to_run.csv"

base_cmd = "java -cp target/sim2apl-dhzw-simulation-1.0-SNAPSHOT-jar-with-dependencies.jar main.java.nl.uu.iss.ga.Simulation" + " " + config +  " " + parametersPath + " --use_random_seed true"


    
# set current directory the folder of Sim2APL so I can execute the jar with the correct paths
if(os.path.basename(os.getcwd()) != 'DHZW-simulation_Sim-2APL'):
    # new_directory = os.path.join(os.getcwd(), '7-simulation-Sim-2APL')
    # new_directory = new_directory.replace('\\', '/')
    os.chdir("C:/Users/ed/Development/dhzw/7-simulation-Sim-2APL") #sorry folks, I'm outta time!

### Main loop to call the simulations

In [26]:
idx_restart = 0 # in case somethings breaks it is easy to continue

In [27]:
time_beginning_all = datetime.now()

for idx, row in tqdm(table.iterrows(), total=table.shape[0]):
    if idx >= idx_restart:
        # repeat each parameter set 5 times
        for iteration in range(0, 5):

            # output file for this iteration
            outputPath = f'--output_file=../8-sensitivity-analysis/output/iterations/output_proportions_{idx}.csv'
            cmd = base_cmd + ' ' + outputPath
            
            # parameterset to use
            arg = f'--parameterset_index={idx}'
            full_command = f'{cmd} {arg}'
            
            time_beginning_iteration = datetime.now()
            
            print('parameter number: ' + str(idx) + ' - iteration: ' + str(iteration) + ' START || Time now: ' + time_beginning_iteration.strftime("%H:%M:%S"))
                                    
            # Capture the output of the Java program
            try:
                output = subprocess.check_output(full_command, stderr=subprocess.STDOUT, universal_newlines=True)
            except subprocess.CalledProcessError as e:
                print(f"Java program exited with non-zero return code: {e.returncode}")
                print(f"Error message: {e.output}")
                exit(1)
                
            time_end_iteration = datetime.now()
         
            print('parameter number: ' + str(idx) + ' - iteration: ' + str(iteration) + ' END   || Time now: ' + time_end_iteration.strftime("%H:%M:%S") + ' | Duration iteration: ' + calculate_elapsed_time(time_beginning_iteration, time_end_iteration) + ' | Elapsed from start: ' + calculate_elapsed_time(time_beginning_all, time_end_iteration))

  0%|          | 0/66 [00:00<?, ?it/s]

parameter number: 0 - iteration: 0 START || Time now: 19:49:38
parameter number: 0 - iteration: 0 END   || Time now: 19:49:53 | Duration iteration: 00:00:14 | Elapsed from start: 00:00:14
parameter number: 0 - iteration: 1 START || Time now: 19:49:53
parameter number: 0 - iteration: 1 END   || Time now: 19:50:09 | Duration iteration: 00:00:15 | Elapsed from start: 00:00:30
parameter number: 0 - iteration: 2 START || Time now: 19:50:09
parameter number: 0 - iteration: 2 END   || Time now: 19:50:25 | Duration iteration: 00:00:15 | Elapsed from start: 00:00:46
parameter number: 0 - iteration: 3 START || Time now: 19:50:25
parameter number: 0 - iteration: 3 END   || Time now: 19:50:40 | Duration iteration: 00:00:15 | Elapsed from start: 00:01:01
parameter number: 0 - iteration: 4 START || Time now: 19:50:40


  2%|▏         | 1/66 [01:17<1:23:49, 77.38s/it]

parameter number: 0 - iteration: 4 END   || Time now: 19:50:56 | Duration iteration: 00:00:15 | Elapsed from start: 00:01:17
parameter number: 1 - iteration: 0 START || Time now: 19:50:56
parameter number: 1 - iteration: 0 END   || Time now: 19:51:11 | Duration iteration: 00:00:15 | Elapsed from start: 00:01:32
parameter number: 1 - iteration: 1 START || Time now: 19:51:11
parameter number: 1 - iteration: 1 END   || Time now: 19:51:27 | Duration iteration: 00:00:15 | Elapsed from start: 00:01:48
parameter number: 1 - iteration: 2 START || Time now: 19:51:27
parameter number: 1 - iteration: 2 END   || Time now: 19:51:42 | Duration iteration: 00:00:15 | Elapsed from start: 00:02:03
parameter number: 1 - iteration: 3 START || Time now: 19:51:42
parameter number: 1 - iteration: 3 END   || Time now: 19:51:58 | Duration iteration: 00:00:15 | Elapsed from start: 00:02:19
parameter number: 1 - iteration: 4 START || Time now: 19:51:58


  3%|▎         | 2/66 [02:34<1:22:24, 77.26s/it]

...

parameter number: 63 - iteration: 4 END   || Time now: 21:10:42 | Duration iteration: 00:00:15 | Elapsed from start: 01:21:03
parameter number: 64 - iteration: 0 START || Time now: 21:10:42
parameter number: 64 - iteration: 0 END   || Time now: 21:10:57 | Duration iteration: 00:00:14 | Elapsed from start: 01:21:18
parameter number: 64 - iteration: 1 START || Time now: 21:10:57
parameter number: 64 - iteration: 1 END   || Time now: 21:11:12 | Duration iteration: 00:00:15 | Elapsed from start: 01:21:33
parameter number: 64 - iteration: 2 START || Time now: 21:11:12
parameter number: 64 - iteration: 2 END   || Time now: 21:11:27 | Duration iteration: 00:00:15 | Elapsed from start: 01:21:49
parameter number: 64 - iteration: 3 START || Time now: 21:11:27
parameter number: 64 - iteration: 3 END   || Time now: 21:11:43 | Duration iteration: 00:00:15 | Elapsed from start: 01:22:04
parameter number: 64 - iteration: 4 START || Time now: 21:11:43


 98%|█████████▊| 65/66 [1:22:19<01:15, 75.93s/it]

parameter number: 64 - iteration: 4 END   || Time now: 21:11:58 | Duration iteration: 00:00:15 | Elapsed from start: 01:22:19
parameter number: 65 - iteration: 0 START || Time now: 21:11:58
parameter number: 65 - iteration: 0 END   || Time now: 21:12:12 | Duration iteration: 00:00:14 | Elapsed from start: 01:22:33
parameter number: 65 - iteration: 1 START || Time now: 21:12:12
parameter number: 65 - iteration: 1 END   || Time now: 21:12:27 | Duration iteration: 00:00:15 | Elapsed from start: 01:22:49
parameter number: 65 - iteration: 2 START || Time now: 21:12:27
parameter number: 65 - iteration: 2 END   || Time now: 21:12:42 | Duration iteration: 00:00:14 | Elapsed from start: 01:23:03
parameter number: 65 - iteration: 3 START || Time now: 21:12:42
parameter number: 65 - iteration: 3 END   || Time now: 21:12:57 | Duration iteration: 00:00:14 | Elapsed from start: 01:23:18
parameter number: 65 - iteration: 4 START || Time now: 21:12:57


100%|██████████| 66/66 [1:23:33<00:00, 75.96s/it]

parameter number: 65 - iteration: 4 END   || Time now: 21:13:12 | Duration iteration: 00:00:14 | Elapsed from start: 01:23:33
